# Epoch 4 MCA Before / After

This notebook uses the ICA reference workflow and applies MCA-style trend/transient separation to the target IC3.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = ROOT / 'src'
sys.path.insert(0, str(SRC))

from io_utils import load_eeg_set
from ica_utils import fir_filter, compute_ica_activations, plot_ic_epoch
from mca_utils import split_trend_transient

DATASET = ROOT / 'dataset'
SET_FILE = DATASET / 'without_eog_channels.set'
ICA_FILE = DATASET / 'without_eog_channels_Filter_ExtRunica_ICA.set'

In [ ]:
meta, epochs_raw = load_eeg_set(SET_FILE)
epochs_filt = fir_filter(epochs_raw)
ica = compute_ica_activations(epochs_filt, meta, ICA_FILE)

ic3 = ica['ic_all'][3 - 1, 2, :]  # epoch 4, IC3

In [ ]:
mca = split_trend_transient(ic3, max_iter=60, lam_start=1.0, lam_end=0.05)
clean_ic3 = mca.trend
len(mca.residual_history), mca.residual_history[-1]

In [ ]:
t = np.arange(meta['pnts']) / meta['srate']
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(t, ic3, label='IC3 before', color='black', lw=0.8)
ax.plot(t, clean_ic3, label='IC3 after MCA', color='red', lw=1.0)
ax.set_title('Epoch 4 IC3 before/after MCA')
ax.set_xlabel('Time (s)')
ax.legend()
fig.tight_layout()
fig